In [1]:
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

canonical_df = pd.read_pickle(
    "/content/drive/MyDrive/router_project_canonical.pkl"
)

splits = np.load(
    "/content/drive/MyDrive/router_split_indices.npz"
)

distilbert_val_pred = np.load(
    "/content/drive/MyDrive/distilbert_val_predictions.npy"
)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

print("Dataset:", canonical_df.shape)
print("DistilBERT val predictions:", distilbert_val_pred.shape)

Mounted at /content/drive
Dataset: (22962, 26)
DistilBERT val predictions: (2296, 11)


In [2]:
MODEL_COLS = [
    "WizardLM/WizardLM-13B-V1.2",
    "claude-instant-v1",
    "claude-v1",
    "claude-v2",
    "gpt-3.5-turbo-1106",
    "gpt-4-1106-preview",
    "meta/code-llama-instruct-34b-chat",
    "meta/llama-2-70b-chat",
    "mistralai/mistral-7b-chat",
    "mistralai/mixtral-8x7b-chat",
    "zero-one-ai/Yi-34B-Chat"
]

Y = pd.DataFrame({
    model: canonical_df[f"{model}|performance"]
    for model in MODEL_COLS
})

C = pd.DataFrame({
    model: canonical_df[f"{model}|cost"]
    for model in MODEL_COLS
})

Y_train = Y.iloc[train_idx].reset_index(drop=True)
Y_val = Y.iloc[val_idx].reset_index(drop=True)

C_train = C.iloc[train_idx].reset_index(drop=True)
C_val = C.iloc[val_idx].reset_index(drop=True)

print(Y_train.shape, Y_val.shape)
print(C_train.shape, C_val.shape)

(18369, 11) (2296, 11)
(18369, 11) (2296, 11)


In [3]:
mean_train_cost = C_train.mean().to_numpy()

for model, cost in zip(MODEL_COLS, mean_train_cost):
    print(model, "->", cost)

WizardLM/WizardLM-13B-V1.2 -> 7.9269856e-05
claude-instant-v1 -> 0.00026399308
claude-v1 -> 0.0023590098
claude-v2 -> 0.0027059794
gpt-3.5-turbo-1106 -> 0.00026386313
gpt-4-1106-preview -> 0.003713676
meta/code-llama-instruct-34b-chat -> 0.00018116446
meta/llama-2-70b-chat -> 0.00021006948
mistralai/mistral-7b-chat -> 4.8790404e-05
mistralai/mixtral-8x7b-chat -> 0.00014154297
zero-one-ai/Yi-34B-Chat -> 0.00019933419


In [4]:
def route_predictions(
    predictions,
    model_costs,
    threshold
):
    choices = []

    cost_order = np.argsort(model_costs)

    for scores in predictions:

        eligible = [
            k
            for k in cost_order
            if scores[k] >= threshold
        ]

        if len(eligible) > 0:
            selected = eligible[0]

        else:
            selected = int(
                np.argmax(scores)
            )

        choices.append(selected)

    return np.array(choices)

In [5]:
thresholds = np.arange(
    0.10,
    0.96,
    0.05
)

Y_val_np = Y_val.to_numpy()
C_val_np = C_val.to_numpy()

distilbert_results = []

for theta in thresholds:

    choices = route_predictions(
        distilbert_val_pred,
        mean_train_cost,
        theta
    )

    rows = np.arange(len(choices))

    realised_performance = Y_val_np[
        rows,
        choices
    ]

    realised_cost = C_val_np[
        rows,
        choices
    ]

    distilbert_results.append({
        "threshold": theta,
        "mean_performance": realised_performance.mean(),
        "mean_cost": realised_cost.mean()
    })

distilbert_threshold_results = pd.DataFrame(
    distilbert_results
)

display(distilbert_threshold_results)

,threshold,mean_performance,mean_cost
0,0.10,0.325893,0.000048
1,0.15,0.326328,0.000048
2,0.20,0.331555,0.000050
3,0.25,0.356381,0.000056
4,0.30,0.400370,0.000072
5,0.35,0.447844,0.000108
6,0.40,0.498367,0.000151
7,0.45,0.527983,0.000226
8,0.50,0.573062,0.000347
9,0.55,0.610083,0.000544


In [6]:
val_fixed_summary = pd.DataFrame({
    "mean_performance": Y_val.mean(),
    "mean_cost": C_val.mean()
})

val_fixed_summary = val_fixed_summary.sort_values(
    "mean_performance",
    ascending=False
)

best_fixed_model = (
    val_fixed_summary["mean_performance"].idxmax()
)

best_fixed_performance = (
    val_fixed_summary.loc[
        best_fixed_model,
        "mean_performance"
    ]
)

best_fixed_cost = (
    val_fixed_summary.loc[
        best_fixed_model,
        "mean_cost"
    ]
)

cheapest_fixed_model = (
    val_fixed_summary["mean_cost"].idxmin()
)

print("Best fixed:")
print(best_fixed_model)
print(best_fixed_performance)
print(best_fixed_cost)

print("\nCheapest fixed:")
print(cheapest_fixed_model)
print(
    val_fixed_summary.loc[
        cheapest_fixed_model,
        "mean_performance"
    ]
)
print(
    val_fixed_summary.loc[
        cheapest_fixed_model,
        "mean_cost"
    ]
)

Best fixed:
gpt-4-1106-preview
0.77504355
0.003637931

Cheapest fixed:
mistralai/mistral-7b-chat
0.31587544
4.7787453e-05


In [7]:
EPSILON = 0.01

target_performance = (
    best_fixed_performance
    - EPSILON
)

acceptable = distilbert_threshold_results[
    distilbert_threshold_results["mean_performance"]
    >= target_performance
].copy()

if len(acceptable) > 0:

    distilbert_best_point = acceptable.loc[
        acceptable["mean_cost"].idxmin()
    ]

    print(
        "DistilBERT reached the target."
    )

else:

    distilbert_best_point = (
        distilbert_threshold_results.loc[
            distilbert_threshold_results[
                "mean_performance"
            ].idxmax()
        ]
    )

    print(
        "DistilBERT did NOT reach the 1-point target."
    )

print("\nChosen operating point:")
print(distilbert_best_point)

DistilBERT reached the target.

Chosen operating point:
threshold           0.95000
mean_performance    0.76851
mean_cost           0.00285
Name: 17, dtype: float64


In [8]:
oracle_scores = []
oracle_costs = []

for i in range(len(Y_val_np)):

    scores = Y_val_np[i]
    costs = C_val_np[i]

    best_score = scores.max()

    candidates = np.where(
        scores == best_score
    )[0]

    selected = candidates[
        np.argmin(
            costs[candidates]
        )
    ]

    oracle_scores.append(
        scores[selected]
    )

    oracle_costs.append(
        costs[selected]
    )

oracle_performance = np.mean(
    oracle_scores
)

oracle_cost = np.mean(
    oracle_costs
)

print("Oracle performance:", oracle_performance)
print("Oracle cost:", oracle_cost)

Oracle performance: 0.8941638
Oracle cost: 0.00027800963


In [9]:
TFIDF_PERFORMANCE = 0.731816
TFIDF_COST = 0.002157
TFIDF_THRESHOLD = 0.95

In [10]:
TFIDF_PERFORMANCE = 0.731816
TFIDF_COST = 0.002157
TFIDF_THRESHOLD = 0.95

In [11]:
comparison = pd.DataFrame([
    {
        "system": "Cheapest Fixed",
        "performance":
            val_fixed_summary.loc[
                cheapest_fixed_model,
                "mean_performance"
            ],
        "cost":
            val_fixed_summary.loc[
                cheapest_fixed_model,
                "mean_cost"
            ]
    },

    {
        "system": "Best Fixed",
        "performance": best_fixed_performance,
        "cost": best_fixed_cost
    },

    {
        "system": "TF-IDF Router",
        "performance": TFIDF_PERFORMANCE,
        "cost": TFIDF_COST
    },

    {
        "system": "DistilBERT Router",
        "performance": float(
            distilbert_best_point[
                "mean_performance"
            ]
        ),
        "cost": float(
            distilbert_best_point[
                "mean_cost"
            ]
        )
    },

    {
        "system": "Oracle",
        "performance": oracle_performance,
        "cost": oracle_cost
    }
])

display(comparison)

,system,performance,cost
0,Cheapest Fixed,0.315875,0.000048
1,Best Fixed,0.775044,0.003638
2,TF-IDF Router,0.731816,0.002157
3,DistilBERT Router,0.768510,0.002850
4,Oracle,0.894164,0.000278


In [12]:
distilbert_cost = float(
    distilbert_best_point["mean_cost"]
)

distilbert_performance = float(
    distilbert_best_point["mean_performance"]
)

cost_saving_vs_best = (
    1
    - distilbert_cost / best_fixed_cost
) * 100

performance_gap = (
    best_fixed_performance
    - distilbert_performance
) * 100

print(
    "DistilBERT cost saving vs best fixed:",
    round(cost_saving_vs_best, 2),
    "%"
)

print(
    "Performance gap vs best fixed:",
    round(performance_gap, 2),
    "percentage points"
)

DistilBERT cost saving vs best fixed: 21.66 %
Performance gap vs best fixed: 0.65 percentage points


In [13]:
print("===== ROUTER COMPARISON =====")

print(
    "TF-IDF performance:",
    TFIDF_PERFORMANCE
)

print(
    "TF-IDF cost:",
    TFIDF_COST
)

print()

print(
    "DistilBERT performance:",
    distilbert_performance
)

print(
    "DistilBERT cost:",
    distilbert_cost
)

print()

print(
    "Performance improvement over TF-IDF:",
    distilbert_performance
    - TFIDF_PERFORMANCE
)

print(
    "Cost difference vs TF-IDF:",
    distilbert_cost
    - TFIDF_COST
)

===== ROUTER COMPARISON =====
TF-IDF performance: 0.731816
TF-IDF cost: 0.002157

DistilBERT performance: 0.7685104608535767
DistilBERT cost: 0.0028499444015324116

Performance improvement over TF-IDF: 0.03669446085357664
Cost difference vs TF-IDF: 0.0006929444015324115


In [14]:
fine_thresholds = np.round(
    np.arange(0.90, 1.001, 0.01),
    2
)

fine_results = []

for theta in fine_thresholds:

    choices = route_predictions(
        distilbert_val_pred,
        mean_train_cost,
        theta
    )

    rows = np.arange(len(choices))

    realised_performance = Y_val_np[
        rows,
        choices
    ]

    realised_cost = C_val_np[
        rows,
        choices
    ]

    fine_results.append({
        "threshold": theta,
        "mean_performance": realised_performance.mean(),
        "mean_cost": realised_cost.mean()
    })

fine_results = pd.DataFrame(fine_results)

display(fine_results)

,threshold,mean_performance,mean_cost
0,0.90,0.758057,0.002755
1,0.91,0.758493,0.002772
2,0.92,0.763284,0.002795
3,0.93,0.763719,0.002816
4,0.94,0.767639,0.002836
5,0.95,0.768510,0.002850
6,0.96,0.768510,0.002868
7,0.97,0.768946,0.002883
8,0.98,0.771995,0.002896
9,0.99,0.773737,0.002917


In [15]:
EPSILON = 0.01

target_performance = (
    best_fixed_performance
    - EPSILON
)

fine_acceptable = fine_results[
    fine_results["mean_performance"]
    >= target_performance
].copy()

FINAL_THRESHOLD_ROW = fine_acceptable.loc[
    fine_acceptable["mean_cost"].idxmin()
]

FINAL_THRESHOLD = float(
    FINAL_THRESHOLD_ROW["threshold"]
)

print("Required performance:", target_performance)

print("\nFinal validation operating point:")
print(FINAL_THRESHOLD_ROW)

print("\nFINAL THRESHOLD:")
print(FINAL_THRESHOLD)

Required performance: 0.76504356

Final validation operating point:
threshold           0.940000
mean_performance    0.767639
mean_cost           0.002836
Name: 4, dtype: float64

FINAL THRESHOLD:
0.94


In [16]:
np.save(
    "/content/drive/MyDrive/distilbert_final_threshold.npy",
    np.array([FINAL_THRESHOLD])
)

fine_results.to_csv(
    "/content/drive/MyDrive/distilbert_fine_threshold_results.csv",
    index=False
)

print("Final threshold saved.")

Final threshold saved.
